In [1]:
import sys
import json
import random
from pathlib import Path
from typing import Any

import numpy as np
import torch
from sklearn.model_selection import train_test_split

from transformers import AutoImageProcessor, AutoTokenizer

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from src.emotion_reasoning.config import load_experiment_config
from src.emotion_reasoning.datasets import build_dataset
from src.emotion_reasoning.evaluation.ablation import run_ablation_suite
from src.emotion_reasoning.evaluation.attention_viz import aggregate_cross_attention, overlay_attention_on_image
from src.emotion_reasoning.modeling.multimodal_model import MultimodalEmotionModel
from src.emotion_reasoning.training.trainer import evaluate_model, train_experiment
from src.emotion_reasoning.utils.io import load_checkpoint, load_records, save_json, save_records

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_ROOT = PROJECT_ROOT / "caer_dataset" / "CAER-S"
WORKDIR = PROJECT_ROOT / "notebook_outputs" / "risetv1_qwen"
WORKDIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT exists:", DATASET_ROOT.exists())
print("WORKDIR:", WORKDIR)
print("DEVICE:", DEVICE)

/home/agung/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /home/agung/riset
DATASET_ROOT exists: True
WORKDIR: /home/agung/riset/notebook_outputs/risetv1_qwen
DEVICE: cuda:1


In [ ]:
ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
VAL_RATIO = 0.2

ANNOTATION_PATH = WORKDIR / "annotations" / "caers_annotations_with_val.jsonl"
BUILD_BASE_ANNOTATION = False


def _stable_sample_id(rel_path: str, split_name: str) -> str:
    return f"{split_name}__{rel_path.replace('/', '__').replace('.', '_')}"


def _collect_split(split_name: str) -> list[dict[str, Any]]:
    split_dir = DATASET_ROOT / split_name
    if not split_dir.exists():
        return []

    rows: list[dict[str, Any]] = []
    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        label_name = cls_dir.name
        for img_path in sorted(cls_dir.rglob("*")):
            if not img_path.is_file() or img_path.suffix.lower() not in ALLOWED_EXTS:
                continue

            rel_path = img_path.relative_to(DATASET_ROOT).as_posix()
            rows.append(
                {
                    "sample_id": _stable_sample_id(rel_path, split_name),
                    "image_path": rel_path,
                    "labels": label_name,
                    "split": split_name,
                    "bbox": None,
                }
            )
    return rows


if BUILD_BASE_ANNOTATION:
    train_raw = _collect_split("train")
    test_raw = _collect_split("test")

    if len(train_raw) == 0 or len(test_raw) == 0:
        raise FileNotFoundError(
            "Dataset CAER-S tidak ditemukan lengkap di caer_dataset/CAER-S/{train,test}."
        )

    indices = np.arange(len(train_raw))
    strat_labels = [row["labels"] for row in train_raw]
    train_idx, val_idx = train_test_split(
        indices,
        test_size=VAL_RATIO,
        random_state=SEED,
        stratify=strat_labels,
    )

    train_records: list[dict[str, Any]] = []
    val_records: list[dict[str, Any]] = []
    for i in train_idx:
        row = dict(train_raw[int(i)])
        row["split"] = "train"
        row["sample_id"] = _stable_sample_id(row["image_path"], "train")
        train_records.append(row)

    for i in val_idx:
        row = dict(train_raw[int(i)])
        row["split"] = "val"
        row["sample_id"] = _stable_sample_id(row["image_path"], "val")
        val_records.append(row)

    test_records: list[dict[str, Any]] = []
    for row in test_raw:
        copied = dict(row)
        copied["split"] = "test"
        copied["sample_id"] = _stable_sample_id(copied["image_path"], "test")
        test_records.append(copied)

    all_records = train_records + val_records + test_records
    save_records(ANNOTATION_PATH, all_records)

    print("Annotation saved:", ANNOTATION_PATH)
    print(
        "Split sizes:",
        {
            "train": len(train_records),
            "val": len(val_records),
            "test": len(test_records),
            "total": len(all_records),
        },
    )
else:
    print("BUILD_BASE_ANNOTATION=False, skip rebuild annotation CAER-S.")
    if ANNOTATION_PATH.exists():
        existing_records = load_records(ANNOTATION_PATH)
        existing_split_sizes: dict[str, int] = {}
        for row in existing_records:
            split_name = str(row.get("split", "unknown")).strip()
            existing_split_sizes[split_name] = existing_split_sizes.get(split_name, 0) + 1
        print("Existing annotation:", ANNOTATION_PATH)
        print("Existing split sizes:", existing_split_sizes)
    else:
        print("Catatan: annotation belum ada, tetapi pipeline training tetap bisa lanjut dari Cell 3.")

Annotation saved: /home/agung/riset/notebook_outputs/risetv1_qwen/annotations/caers_annotations_with_val.jsonl
Class names: ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
Split sizes: {'train': 39205, 'val': 9802, 'test': 20992, 'total': 69999}


In [3]:
# Stage-1 pseudo labeling tidak dijalankan di notebook ini.
# Notebook hanya menyiapkan data stage-2 dari file pseudo-label yang sudah tersedia.
PSEUDO_LABEL_SOURCE_PATH = PROJECT_ROOT / "notebook_outputs" / "risetv1_qwen" / "stage1_pseudo_labels_qwen.jsonl"
STAGE1_OUTPUT_PATH = WORKDIR / "stage1_pseudo_labels_qwen_train10k_ready.jsonl"

TARGET_TOTAL_PSEUDO = 10_000
VAL_RATIO_FROM_SELECTED = 0.1
TEST_RATIO_FROM_SELECTED = 0.1

if not PSEUDO_LABEL_SOURCE_PATH.exists():
    raise FileNotFoundError(f"Pseudo-label source tidak ditemukan: {PSEUDO_LABEL_SOURCE_PATH}")

if VAL_RATIO_FROM_SELECTED <= 0 or TEST_RATIO_FROM_SELECTED <= 0:
    raise ValueError("VAL_RATIO_FROM_SELECTED dan TEST_RATIO_FROM_SELECTED harus > 0")
if (VAL_RATIO_FROM_SELECTED + TEST_RATIO_FROM_SELECTED) >= 1.0:
    raise ValueError("VAL_RATIO_FROM_SELECTED + TEST_RATIO_FROM_SELECTED harus < 1")

source_rows = load_records(PSEUDO_LABEL_SOURCE_PATH)
if len(source_rows) == 0:
    raise ValueError("Pseudo-label source kosong.")

if len(source_rows) < TARGET_TOTAL_PSEUDO:
    print(
        f"Peringatan: source hanya berisi {len(source_rows)} row, lebih kecil dari TARGET_TOTAL_PSEUDO={TARGET_TOTAL_PSEUDO}."
    )

selected_total = min(TARGET_TOTAL_PSEUDO, len(source_rows))
source_indices = np.arange(len(source_rows))
source_labels = [str(row.get("labels", "")).strip() for row in source_rows]

if selected_total < len(source_rows):
    try:
        selected_idx, _ = train_test_split(
            source_indices,
            train_size=selected_total,
            random_state=SEED,
            stratify=source_labels,
        )
    except ValueError:
        selected_idx, _ = train_test_split(
            source_indices,
            train_size=selected_total,
            random_state=SEED,
            stratify=None,
        )
    selected_idx = sorted(int(i) for i in selected_idx)
    selected_rows = [dict(source_rows[i]) for i in selected_idx]
else:
    selected_rows = [dict(row) for row in source_rows[:selected_total]]

usable_rows: list[dict[str, Any]] = []
dropped_empty_text = 0
dropped_missing_image = 0

for row in selected_rows:
    image_ref = Path(str(row.get("image_path", "")))
    image_path = image_ref if image_ref.is_absolute() else DATASET_ROOT / image_ref
    if not image_path.exists():
        dropped_missing_image += 1
        continue

    pseudo_text = str(row.get("semantic_pseudo_label", "")).strip()
    if not pseudo_text:
        dropped_empty_text += 1
        continue

    updated = dict(row)
    if str(updated.get("labels", "")).strip() == "Anger":
        updated["labels"] = "Angry"
    usable_rows.append(updated)

if len(usable_rows) < 20:
    raise ValueError(
        "Jumlah row pseudo-label yang usable terlalu sedikit setelah filtering. "
        f"usable={len(usable_rows)}"
    )

usable_indices = np.arange(len(usable_rows))
usable_labels = [str(row.get("labels", "")).strip() for row in usable_rows]
holdout_ratio = VAL_RATIO_FROM_SELECTED + TEST_RATIO_FROM_SELECTED

try:
    train_idx, holdout_idx = train_test_split(
        usable_indices,
        test_size=holdout_ratio,
        random_state=SEED,
        stratify=usable_labels,
    )
except ValueError:
    train_idx, holdout_idx = train_test_split(
        usable_indices,
        test_size=holdout_ratio,
        random_state=SEED,
        stratify=None,
    )

holdout_labels = [usable_labels[int(i)] for i in holdout_idx]
test_share_in_holdout = TEST_RATIO_FROM_SELECTED / holdout_ratio

try:
    val_idx, test_idx = train_test_split(
        holdout_idx,
        test_size=test_share_in_holdout,
        random_state=SEED,
        stratify=holdout_labels,
    )
except ValueError:
    val_idx, test_idx = train_test_split(
        holdout_idx,
        test_size=test_share_in_holdout,
        random_state=SEED,
        stratify=None,
    )


def _stable_sample_id_for_stage2(image_path: str, split_name: str) -> str:
    rel_path = str(image_path).replace("\\", "/")
    return f"{split_name}__{rel_path.replace('/', '__').replace('.', '_')}"


train_records: list[dict[str, Any]] = []
val_records: list[dict[str, Any]] = []
test_records: list[dict[str, Any]] = []

for i in train_idx:
    row = dict(usable_rows[int(i)])
    row["split"] = "train"
    row["sample_id"] = _stable_sample_id_for_stage2(str(row["image_path"]), "train")
    train_records.append(row)

for i in val_idx:
    row = dict(usable_rows[int(i)])
    row["split"] = "val"
    row["sample_id"] = _stable_sample_id_for_stage2(str(row["image_path"]), "val")
    val_records.append(row)

for i in test_idx:
    row = dict(usable_rows[int(i)])
    row["split"] = "test"
    row["sample_id"] = _stable_sample_id_for_stage2(str(row["image_path"]), "test")
    test_records.append(row)

stage2_records = train_records + val_records + test_records
save_records(STAGE1_OUTPUT_PATH, stage2_records)

CLASS_NAMES = sorted({str(row["labels"]).strip() for row in stage2_records})
print("Pseudo-label source:", PSEUDO_LABEL_SOURCE_PATH)
print("Stage-2 ready file:", STAGE1_OUTPUT_PATH)
print("Rows source:", len(source_rows))
print("Rows selected:", len(selected_rows))
print("Rows usable:", len(usable_rows))
print("Dropped empty text:", dropped_empty_text)
print("Dropped missing image:", dropped_missing_image)
print(
    "Split sizes:",
    {
        "train": len(train_records),
        "val": len(val_records),
        "test": len(test_records),
        "total": len(stage2_records),
    },
)
print("Class names:", CLASS_NAMES)

Pseudo-label source: /home/agung/riset/notebook_outputs/risetv1_qwen/stage1_pseudo_labels_qwen.jsonl
Stage-2 ready file: /home/agung/riset/notebook_outputs/risetv1_qwen/stage1_pseudo_labels_qwen_train10k_ready.jsonl
Rows source: 10000
Rows selected: 10000
Rows usable: 10000
Dropped empty text: 0
Dropped missing image: 0
Split sizes: {'train': 8000, 'val': 1000, 'test': 1000, 'total': 10000}
Class names: ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


In [4]:
# Pseudo labeling stage-1 dinonaktifkan di notebook ini.
# File STAGE1_OUTPUT_PATH sudah disiapkan pada Cell 3 dari pseudo-label yang sudah ada.
if not STAGE1_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Stage-2 input belum ada: {STAGE1_OUTPUT_PATH}. Jalankan Cell 3 terlebih dahulu."
    )

stage2_input_rows = load_records(STAGE1_OUTPUT_PATH)
stage2_split_sizes: dict[str, int] = {}
for row in stage2_input_rows:
    split_name = str(row.get("split", "unknown")).strip()
    stage2_split_sizes[split_name] = stage2_split_sizes.get(split_name, 0) + 1

print("Stage-1 generation: SKIPPED (external pseudo-label file)")
print("Stage-2 input path:", STAGE1_OUTPUT_PATH)
print("Stage-2 input rows:", len(stage2_input_rows))
print("Stage-2 split sizes:", stage2_split_sizes)
print("Class names:", CLASS_NAMES)

Stage-1 generation: SKIPPED (external pseudo-label file)
Stage-2 input path: /home/agung/riset/notebook_outputs/risetv1_qwen/stage1_pseudo_labels_qwen_train10k_ready.jsonl
Stage-2 input rows: 10000
Stage-2 split sizes: {'train': 8000, 'val': 1000, 'test': 1000}
Class names: ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


In [5]:
CONFIG_PATH = WORKDIR / "configs" / "caers_qwen_qformer.json"
OUTPUT_DIR = WORKDIR / "stage2_qformer"

config_payload = {
    "experiment_name": "caers_qwen_qformer_notebook",
    "seed": SEED,
    "dataset": {
        "name": "caer-s",
        "annotation_path": str(STAGE1_OUTPUT_PATH),
        "image_root": str(DATASET_ROOT),
        "task_type": "singlelabel",
        "image_column": "image_path",
        "label_column": "labels",
        "split_column": "split",
        "sample_id_column": "sample_id",
        "bbox_column": "bbox",
        "pseudo_label_column": "semantic_pseudo_label",
        "train_split": "train",
        "val_split": "val",
        "test_split": "test",
        "num_workers": 2,
        "max_text_length": 96,
        "class_names": CLASS_NAMES,
    },
    "model": {
        "vision_encoder_name": "openai/clip-vit-base-patch32",
        "text_encoder_name": "roberta-base",
        "num_queries": 32,
        "qformer_hidden_size": 512,
        "qformer_num_layers": 4,
        "qformer_num_heads": 8,
        "dropout": 0.3,
        "fusion_mode": "multimodal",
        "freeze_vision_encoder": False,
        "freeze_text_encoder": False,
    },
    "training": {
        "batch_size": 8,
        "epochs": 12,
        "gradient_clip_norm": 1.0,
        "mixed_precision": True,
        "early_stopping_patience": 4,
        "weight_decay": 0.05,
        "vision_lr": 1e-5,
        "text_lr": 1e-4,
        "qformer_lr": 1e-4,
        "head_lr": 1e-3,
        "output_dir": str(OUTPUT_DIR),
    },
}

save_json(CONFIG_PATH, config_payload)
experiment_config = load_experiment_config(CONFIG_PATH)

print("Config saved:", CONFIG_PATH)
print(json.dumps(config_payload, indent=2))

Config saved: /home/agung/riset/notebook_outputs/risetv1_qwen/configs/caers_qwen_qformer.json
{
  "experiment_name": "caers_qwen_qformer_notebook",
  "seed": 42,
  "dataset": {
    "name": "caer-s",
    "annotation_path": "/home/agung/riset/notebook_outputs/risetv1_qwen/stage1_pseudo_labels_qwen_train10k_ready.jsonl",
    "image_root": "/home/agung/riset/caer_dataset/CAER-S",
    "task_type": "singlelabel",
    "image_column": "image_path",
    "label_column": "labels",
    "split_column": "split",
    "sample_id_column": "sample_id",
    "bbox_column": "bbox",
    "pseudo_label_column": "semantic_pseudo_label",
    "train_split": "train",
    "val_split": "val",
    "test_split": "test",
    "num_workers": 2,
    "max_text_length": 96,
    "class_names": [
      "Angry",
      "Disgust",
      "Fear",
      "Happy",
      "Neutral",
      "Sad",
      "Surprise"
    ]
  },
  "model": {
    "vision_encoder_name": "openai/clip-vit-base-patch32",
    "text_encoder_name": "roberta-base",


In [ ]:
TRAIN_STAGE2 = True

if not STAGE1_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Stage-1 output belum ada: {STAGE1_OUTPUT_PATH}. Jalankan Cell 3 dulu."
    )

if TRAIN_STAGE2:
    train_results = train_experiment(experiment_config)
    print("Training finished.")
    print(json.dumps(train_results, indent=2))
else:
    train_results = None
    print("TRAIN_STAGE2=False, skip training.")

BEST_CHECKPOINT = Path(experiment_config.training.output_dir) / "best.pt"
RESULTS_JSON = Path(experiment_config.training.output_dir) / "results.json"
HISTORY_JSON = Path(experiment_config.training.output_dir) / "history.json"

print("Best checkpoint:", BEST_CHECKPOINT, "| exists:", BEST_CHECKPOINT.exists())
print("Results file:", RESULTS_JSON, "| exists:", RESULTS_JSON.exists())
print("History file:", HISTORY_JSON, "| exists:", HISTORY_JSON.exists())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 62969.94it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEX

In [ ]:
RUN_EVAL = True
RUN_ABLATION = False

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best checkpoint belum ditemukan: {BEST_CHECKPOINT}. Jalankan training dulu."
    )

if RUN_EVAL:
    eval_config = load_experiment_config(CONFIG_PATH)
    test_metrics = evaluate_model(
        config=eval_config,
        checkpoint_path=BEST_CHECKPOINT,
        split="test",
        fusion_mode="multimodal",
    )
    print("Test metrics (multimodal):")
    print(json.dumps(test_metrics, indent=2))
else:
    test_metrics = None
    print("RUN_EVAL=False, skip evaluation.")

if RUN_ABLATION:
    ablation_config = load_experiment_config(CONFIG_PATH)
    ablation_summary = run_ablation_suite(ablation_config)
    print("Ablation summary:")
    print(json.dumps(ablation_summary, indent=2))
else:
    ablation_summary = None
    print("RUN_ABLATION=False, skip ablation (set True to run vision/text/multimodal ablation).")

ablation_file = (
    Path(experiment_config.training.output_dir).parent
    / f"{Path(experiment_config.training.output_dir).name}_ablation_summary.json"
)
print("Ablation file:", ablation_file, "| exists:", ablation_file.exists())

In [ ]:
ATTENTION_OUTPUT = WORKDIR / "stage2_qformer" / "attention_overlay_test0.png"
GENERATE_ATTENTION = True

if GENERATE_ATTENTION:
    viz_config = load_experiment_config(CONFIG_PATH)
    test_dataset = build_dataset(viz_config.dataset, split="test")
    if len(test_dataset) == 0:
        raise ValueError("Test dataset kosong, tidak bisa membuat attention overlay.")

    sample = test_dataset[0]
    tokenizer = AutoTokenizer.from_pretrained(viz_config.model.text_encoder_name, use_fast=True)
    image_processor = AutoImageProcessor.from_pretrained(viz_config.model.vision_encoder_name)

    text_inputs = tokenizer(
        [sample["text"] or ""],
        padding=True,
        truncation=True,
        max_length=viz_config.dataset.max_text_length,
        return_tensors="pt",
    )
    image_inputs = image_processor(images=[sample["image"]], return_tensors="pt")

    model_viz = MultimodalEmotionModel(viz_config.model, num_classes=viz_config.num_classes).to(DEVICE)
    checkpoint = load_checkpoint(BEST_CHECKPOINT, map_location=DEVICE)
    model_viz.load_state_dict(checkpoint["model_state_dict"])
    model_viz.eval()

    with torch.inference_mode():
        outputs = model_viz(
            pixel_values=image_inputs["pixel_values"].to(DEVICE),
            input_ids=text_inputs["input_ids"].to(DEVICE),
            attention_mask=text_inputs["attention_mask"].to(DEVICE),
            output_attentions=True,
        )

    attention_grid = aggregate_cross_attention(outputs["cross_attentions"])[0]
    saved_attention = overlay_attention_on_image(
        image=sample["image"],
        attention_grid=attention_grid,
        output_path=ATTENTION_OUTPUT,
        alpha=0.45,
    )
    print("Attention overlay saved:", saved_attention)
else:
    saved_attention = None
    print("GENERATE_ATTENTION=False, skip attention visualization.")

split_names_present: set[str] = set()
if STAGE1_OUTPUT_PATH.exists():
    for row in load_records(STAGE1_OUTPUT_PATH):
        split_names_present.add(str(row.get("split", "")).strip().lower())

alignment_check = {
    "stage0_annotation_has_train_val_test": ANNOTATION_PATH.exists(),
    "stage1_external_pseudo_label_file_used": STAGE1_OUTPUT_PATH.exists(),
    "stage1_generation_skipped_in_notebook": True,
    "stage1_target_total_pseudo_10k": True,
    "stage1_output_column_semantic_pseudo_label": True,
    "stage1_output_has_train_val_test_split": {"train", "val", "test"}.issubset(split_names_present),
    "stage2_qformer_from_repo": True,
    "stage2_differential_lr_optimizer": True,
    "stage2_metrics_map_auc_accuracy": True,
    "ablation_available": True,
    "cross_attention_visualization_available": saved_attention is not None,
}

print("\nAlignment checklist:")
for key, value in alignment_check.items():
    print(f"- {key}: {value}")

print("\nKey artifacts:")
print("- Annotation:", ANNOTATION_PATH)
print("- Stage1 output:", STAGE1_OUTPUT_PATH)
print("- Config:", CONFIG_PATH)
print("- Best checkpoint:", BEST_CHECKPOINT)
print("- Results:", RESULTS_JSON)
print("- Attention:", ATTENTION_OUTPUT)